# 05 · Results and Long-Tail Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Souhaila-DJAFFAL/longtail-recsys/blob/main/notebooks/05_results_and_longtail_analysis.ipynb)

Final evaluation on the **held-out test items** (used here for the first time), answering three questions:

1. **Accuracy.** How do Popularity, BPR-MF and SASRec compare on HR@10 and NDCG@10?
2. **Popularity bias.** How concentrated are their recommendations on popular items, and how well do
   they serve users whose next item is in the long tail?
3. **Mitigation.** How much accuracy does popularity-aware re-ranking cost for a given gain in
   long-tail exposure?

In [ ]:
# --- Setup: works locally (Jupyter) and on Google Colab ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("02_models.ipynb"):
    !git clone -q https://github.com/Souhaila-DJAFFAL/longtail-recsys.git
    %cd longtail-recsys/notebooks
    !pip install -q tabulate

DATASET = os.environ.get("RECSYS_DATASET", "ml-1m")  # "ml-1m" (real) or "synthetic" (fast check)
QUICK = os.environ.get("RECSYS_QUICK", "0") == "1"     # few epochs, for smoke tests only
ARTIFACTS, RESULTS = "../artifacts", "../results"
os.makedirs(ARTIFACTS, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

In [ ]:
%run 02_models.ipynb
%run 03_evaluation_metrics.ipynb

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
data = load_dataset()
with open(f"{ARTIFACTS}/config_{DATASET}.json") as f:
    CONFIG = json.load(f)
K, MAX_LEN = CONFIG["k"], CONFIG["max_len"]

cb, cs = CONFIG["bpr"], CONFIG["sasrec"]
bpr = BPRMF(data.n_users, data.n_items, cb["dim"]).to(DEVICE)
bpr.load_state_dict(torch.load(f"{ARTIFACTS}/bpr_{DATASET}.pt", map_location=DEVICE))
sasrec = SASRec(data.n_items, MAX_LEN, cs["dim"], cs["n_blocks"], cs["n_heads"], cs["dropout"]).to(DEVICE)
sasrec.load_state_dict(torch.load(f"{ARTIFACTS}/sasrec_{DATASET}.pt", map_location=DEVICE))
models = {"Popularity": Popularity(data.item_pop).to(DEVICE), "BPR-MF": bpr, "SASRec": sasrec}
LAMBDAS = (0.0, 0.1, 0.25, 0.5, 0.75, 1.0)

In [ ]:
rows = []
for name, model in models.items():
    lams = (0.0,) if name == "Popularity" else LAMBDAS
    for r in evaluate(model, data, "test", k=K, lambdas=lams, max_len=MAX_LEN, device=DEVICE):
        rows.append({"model": name, **r})
res = pd.DataFrame(rows)
res.to_csv(f"{RESULTS}/results_{DATASET}.csv", index=False)

## 1. Accuracy and popularity bias of the unmodified models

In [ ]:
main = res[res["lambda"] == 0].drop(columns="lambda").set_index("model")
main.style.format("{:.4f}").format("{:.1f}", subset=[f"ARP@{K}"]) \
    .highlight_max(subset=[f"HR@{K}", f"NDCG@{K}", f"NDCG@{K} head", f"NDCG@{K} tail",
                           f"Coverage@{K}", f"TailShare@{K}"], color="#d6e4f0")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.arange(len(main))
w = 0.38
axes[0].bar(x - w / 2, main[f"NDCG@{K} head"], w, label="head-item targets", color="#1f3a68")
axes[0].bar(x + w / 2, main[f"NDCG@{K} tail"], w, label="tail-item targets", color="#e07a2f")
axes[0].set_xticks(x, main.index)
axes[0].set(ylabel=f"NDCG@{K}", title="Who is served well? Accuracy by target popularity")
axes[0].legend()
axes[1].bar(x, main[f"TailShare@{K}"], color="#e07a2f", width=0.55)
axes[1].set_xticks(x, main.index)
axes[1].set(ylabel=f"Share of tail items in top-{K}", title="Exposure given to the long tail")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{RESULTS}/head_vs_tail.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Popularity-aware re-ranking: the accuracy / long-tail trade-off

Each point is one value of $\lambda$; moving right means more long-tail items in the top-10 lists.

In [ ]:
learned = res[res["model"] != "Popularity"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
colors = {"BPR-MF": "#6b8e23", "SASRec": "#1f3a68"}
for name, g in learned.groupby("model"):
    axes[0].plot(g[f"TailShare@{K}"], g[f"NDCG@{K}"], marker="o", color=colors[name], label=name)
    for _, r in g.iterrows():
        axes[0].annotate(f"λ={r['lambda']:g}", (r[f"TailShare@{K}"], r[f"NDCG@{K}"]),
                         textcoords="offset points", xytext=(4, 4), fontsize=7)
    axes[1].plot(g["lambda"], g[f"Coverage@{K}"], marker="o", color=colors[name], label=name)
pop = res[res["model"] == "Popularity"].iloc[0]
axes[0].scatter(pop[f"TailShare@{K}"], pop[f"NDCG@{K}"], marker="x", color="k", label="Popularity")
axes[0].set(xlabel=f"Tail-item share in top-{K}", ylabel=f"NDCG@{K}", title="Accuracy vs. long-tail exposure")
axes[1].set(xlabel="Re-ranking strength λ", ylabel=f"Catalogue coverage@{K}", title="Coverage grows with λ")
for ax in axes:
    ax.legend(); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{RESULTS}/tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cols = ["model", "lambda", f"NDCG@{K}", f"NDCG@{K} tail", f"Coverage@{K}", f"TailShare@{K}", f"ARP@{K}"]
learned[cols].set_index(["model", "lambda"]).style.format("{:.4f}").format("{:.1f}", subset=[f"ARP@{K}"])

## 3. Summary

The cell below writes the key numbers as text (computed, not hand-typed) and saves the tables as
Markdown for the README.

In [ ]:
def pct(a, b):
    return 100 * (a - b) / b

m = main
s, b, p = m.loc["SASRec"], m.loc["BPR-MF"], m.loc["Popularity"]
best_lam = {}
for name, g in learned.groupby("model"):
    base = g[g["lambda"] == 0].iloc[0]
    ok = g[g[f"NDCG@{K}"] >= 0.95 * base[f"NDCG@{K}"]]      # largest lambda costing <= 5% NDCG
    best_lam[name] = (base, ok.sort_values("lambda").iloc[-1])

lines = [
    f"- SASRec reaches NDCG@{K} = {s[f'NDCG@{K}']:.4f}, {pct(s[f'NDCG@{K}'], b[f'NDCG@{K}']):+.1f}% vs. BPR-MF "
    f"and {pct(s[f'NDCG@{K}'], p[f'NDCG@{K}']):+.1f}% vs. Popularity.",
    f"- For users whose target is a tail item, NDCG@{K} is {s[f'NDCG@{K} tail']:.4f} (SASRec) and "
    f"{b[f'NDCG@{K} tail']:.4f} (BPR-MF), against {s[f'NDCG@{K} head']:.4f} and {b[f'NDCG@{K} head']:.4f} for head targets.",
    f"- Tail items make up {s[f'TailShare@{K}']:.1%} (SASRec) and {b[f'TailShare@{K}']:.1%} (BPR-MF) of top-{K} lists; "
    f"catalogue coverage is {s[f'Coverage@{K}']:.1%} and {b[f'Coverage@{K}']:.1%}.",
]
for name, (base, sel) in best_lam.items():
    lines.append(f"- {name} re-ranked with λ = {sel['lambda']:g} keeps ≥95% of its NDCG@{K} "
                 f"({sel[f'NDCG@{K}']:.4f} vs. {base[f'NDCG@{K}']:.4f}) while tail share goes from "
                 f"{base[f'TailShare@{K}']:.1%} to {sel[f'TailShare@{K}']:.1%} and coverage from "
                 f"{base[f'Coverage@{K}']:.1%} to {sel[f'Coverage@{K}']:.1%}.")
summary = "\n".join(lines)
print(summary)

with open(f"{RESULTS}/results_{DATASET}.md", "w") as f:
    f.write(f"### Test results ({DATASET}, full ranking, K={K})\n\n")
    f.write(main.reset_index().to_markdown(index=False, floatfmt=".4f") + "\n\n")
    f.write("### Popularity-aware re-ranking\n\n")
    f.write(learned[cols].to_markdown(index=False, floatfmt=".4f") + "\n\n")
    f.write("### Key findings\n\n" + summary + "\n")
print("\nSaved", f"{RESULTS}/results_{DATASET}.md")

## Discussion

*Write 3–5 sentences here after running on MovieLens-1M, in your own words, for example:*
* *Which model is most accurate, and why sequence order helps on MovieLens.*
* *Whether accuracy gains come mostly from head-item users, and what that means for niche users.*
* *What λ you would choose for a production platform, and what you would test online (A/B) next.*

**Limitations.** Offline leave-one-out evaluation on a single public dataset; one random seed; uniform
negative sampling; re-ranking is a post-hoc baseline rather than a trained debiasing method.
**Natural extensions:** multiple seeds with confidence intervals, a larger dataset (e.g., Amazon reviews),
in-training debiasing (e.g., inverse propensity weighting), and calibrated / personalized re-ranking.